In [5]:
from datetime import datetime, timezone
import json
import os
import boto3
import pandas as pd
import psycopg2
from prefect import flow, task
from src.API.config import datasource_datagrandlyon_api_credentials_base64
from staging_loading import read_json_from_s3, load_data_to_staging_table

ModuleNotFoundError: No module named 'psycopg2'

In [8]:
from botocore.config import Config

s3_client = boto3.client("s3")

bucket_name = os.getenv("AWS_S3_BUCKET")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_s3_url = os.getenv("AWS_S3_URL")

if not bucket_name or not aws_access_key_id or not aws_secret_access_key or not aws_s3_url:
    raise ValueError("AWS_S3_BUCKET, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY and AWS_S3_URL environment variables must be set")

s3_client = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    endpoint_url=f"{aws_s3_url}",
    config=Config(s3={'addressing_style': 'path'})
)

In [15]:
# 20261024 13 58 01
extraction_date = datetime(2026, 10, 24, 13, 58, 1, tzinfo=timezone.utc)  # Replace with your actual extraction date
source_name = "DataGrandLyon"  # Replace with your actual source name
object_name = "VehicleMonitoring"  # Replace with your actual object name
bucket_name = "raw"  # Replace with your actual bucket name

# Format the timestamp and key for S3 read
timestamp = extraction_date.strftime("%Y%m%d%H%M%S")
object_key = f"{source_name}/{object_name}/{timestamp}_{source_name}_{object_name}.json"
print(object_key)

response = s3_client.get_object(
    Bucket=bucket_name,
    Key=object_key
)

json_data = json.loads(response['Body'].read().decode('utf-8'))

DataGrandLyon/VehicleMonitoring/20261024135801_DataGrandLyon_VehicleMonitoring.json


In [19]:
df = pd.json_normalize(json_data)
df

,Siri.ServiceDelivery.ResponseTimestamp,Siri.ServiceDelivery.ProducerRef.value,Siri.ServiceDelivery.ResponseMessageIdentifier.value,Siri.ServiceDelivery.MoreData,Siri.ServiceDelivery.VehicleMonitoringDelivery,Siri.version
0,2026-10-24T11:57:47.34321665Z,MDL,f3f82e63-e476-48ac-ba31-d70fbe168fe6,False,[{'VehicleActivity': [{'ValidUntilTime': '2026...,2.0


In [ ]:
# Get database connection parameters from environment
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "data_lyon_staging")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "")

conn_string = f"host={db_host} port={db_port} dbname={db_name} user={db_user} password={db_password}"